<h1 style="text-align: center;">2026 부산대학교 TECH WEEK:</h1>
<h1 style="text-align: center;">Autonomous Mobile Robot의 Search & Rescue Mission</h1>

<br><br>

<div style="text-align: right; color: gray; font-style: italic;">
김규래&emsp;<br>
kkr.kyurae.kim@gmail.com&emsp;
</div><br>

---
---

## 1. 문제 정의

이번 프로그램에서는 **자율 이동 로봇 (Autonomous Mobile Robot, AMR)** 을 활용하여 미지의 환경을 탐색하고, 주어진 목표 대상을 찾아 안전하게 이동하는 **Search and Rescue** 문제를 다룹니다.

### **AMR이란?**

**AMR(Autonomous Mobile Robot)** 은 주변 환경을 스스로 인식하고, 현재 상황을 판단하여 목적지까지 자율적으로 이동할 수 있는 로봇입니다.

* 주변 환경 인식: LiDAR, Camera 등의 센서를 이용하여 주변 환경과 장애물을 파악합니다.

* 자기 위치 추정: 이동하면서 자신의 위치와 방향을 추정합니다.

* 환경 이해: 주변 정보를 바탕으로 지도를 생성하고 탐색한 공간을 파악합니다.

* 경로 및 행동 결정: 목적지까지 이동하기 위한 경로를 계획하고, 장애물이나 환경 변화에 대응하며 이동합니다.


### **Search and Rescue 문제**

Search and Rescue에서는 로봇이 사전에 환경에 대한 정보를 알 수 없는 상황에서 스스로 주변을 탐색하고 목표 대상을 찾아야 합니다.

로봇은 탐색 과정에서 센서 정보를 지속적으로 수집하고, 이를 이용하여 주변 환경을 이해하고 자신의 위치를 추정합니다. 목표 대상의 위치를 미리 알 수 없기 때문에, 탐색하지 않은 영역을 찾아 이동하며 목표 대상을 발견해야 합니다.

또한 로봇은 목표를 찾는 과정에서 정적 장애물 및 움직이는 사람과 충돌하지 않고 안전하게 이동해야 합니다. 목표 대상을 찾은 후에는 목표 위치까지 이동하고, 최종적으로 시작 지점으로 복귀하는 과정까지 자율적으로 수행해야 합니다.

이러한 기술은 재난 환경뿐만 아니라 다양한 실제 로봇 시스템에 활용될 수 있습니다.

* 재난 및 위험 환경에서의 탐색 및 구조

* Warehouse에서의 물품 탐색 및 Pick and Deliver

* 도심 환경에서 사람과 장애물을 회피하는 Delivery Robot

* 실내 환경에서 특정 사람이나 물체를 탐색하는 Service Robot


### **AMR의 자율주행 파이프라인**

이번 문제에서는 AMR의 전체 자율주행 과정을 다음과 같은 세 단계의 파이프라인으로 이해할 수 있습니다.

<h5 style="text-align: center;">인지 (Perception) → 계획 (Planning) → 행동 (Action)</h5>

* **인지(Perception):** 다양한 센서를 이용하여 환경, 장애물, 목표 대상 등의 데이터를 수집하고 로봇의 위치 및 주변 상황를 파악합니다.

* **계획(Planning):** 인지한 정보를 바탕으로 환경 상황을 분석하고 목적지까지 최적의 경로를 생성하며, 환경 변화에 맞춰 계획을 실시간으로 갱신합니다.

* **행동(Action):** 계획된 경로에 따라 이동 명령을 생성하며, 이동 중 발생하는 장애물이나 환경 변화에 대응합니다.

<br>

이번 해커톤에서는 이 세 가지 과정을 연결하여 미지의 환경에서 스스로 탐색하고, 목표 대상을 발견하여, 안전하게 이동하고 복귀할 수 있는 자율 로봇 시스템을 설계하는 것을 목표로 합니다.

즉, 단순히 주어진 목적지까지 로봇을 이동시키는 것이 아니라, 알지 못하는 환경에서 스스로 정보를 수집하고, 상황을 판단하며, 적절한 행동을 수행하는 것이 이번 문제의 핵심입니다.

---

## 2. Computer Vision 기초

**Computer Vision** 이란, 컴퓨터가 시각적 정보를 사람처럼 해석하고 이해할 수 있도록 만드는 기술입니다.<br>
카메라를 통해 입력된 이미지나 영상에서 사람, 물체, 공간과 같은 의미 있는 정보를 인식하고 분석하는 것이 주요 목표입니다.

우선, 본 섹션에서 사용할 Computer Vision 관련 라이브러리를 설치하도록 합시다.<br>
아래 명령어를 터미널에서 실행하여 필요한 라이브러리를 설치합니다.

```bash
pip install numpy==1.23.5 opencv-python==4.8.0.74 scikit-image==0.19.3
```

In [ ]:
%%bash
pip install numpy==1.23.5 opencv-python==4.8.0.74 scikit-image==0.19.3 matplotlib==3.7.5

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import skimage.data

위 코드가 에러 없이 실행되었다면 설치가 잘 되었으니 다음으로 넘어가도 되겠습니다.

### **A. 이미지 데이터 구조**

이어서 실습에 사용될 테스트 이미지를 불러와 확인해봅시다.

In [ ]:
image = skimage.data.astronaut()

plt.imshow(image)
plt.axis("off");

이미지는 결국 픽셀 '배열'입니다.

배열의 크기는 "H x W x C"이며, [row, col, channel]이라고도 표현합니다.

위 이미지는 RGB 이미지이기에 Channel의 개수는 3입니다.

In [ ]:
image.shape

저희가 사용하는 astronaut 이미지는 세로와 가로가 512 픽셀이며 각 픽셀마다 3가지 값 (RGB)가 저장되어 있습니다.

`image.shape`에서 알 수 있듯이, 이미지는 NumPy 배열입니다.

그럼 NumPy 방식으로 indexing 하여 픽셀 하나만 골라서 확인해볼까요?

In [ ]:
image[0, 0]

위 결과에서 볼 수 있듯이, 픽셀 하나에는 R,G,B 값이 들어있습니다.

각 픽셀의 데이터 타입은 `uint8` 입니다.

색상을 확인해봅시다.

In [ ]:
rgb = np.array([154, 147, 151])

color = np.zeros((1, 1, 3), dtype=np.uint8)
color[0, 0] = rgb

plt.imshow(color)
plt.axis("off");

이미지는 원점 (origin)이 좌측 상단입니다. row가 증가하면 아래로, column이 증가하면 우측으로 이동합니다.

In [ ]:
image[50, 0]

In [ ]:
rgb = np.array([45, 34, 93])

color = np.zeros((1, 1, 3), dtype=np.uint8)
color[0, 0] = rgb

plt.imshow(color)
plt.axis("off");

이번에는 NumPy 방식으로 배열을 slicing 해보면 어떻게 될까요?

In [ ]:
sliced = image[0:100, 0:50]

plt.imshow(sliced)
plt.axis("off");

NumPy slcing을 한다면 이미지를 crop하는 효과를 낼 수 있습니다.

다음으로는 OpenCV 기능을 사용하여 이미지를 회전/반전시켜 봅시다.

In [ ]:
plt.figure(figsize=(15,6))

plt.subplot(1,4,1), plt.imshow(image)
plt.subplot(1,4,2), plt.imshow(cv2.rotate(image.copy(), cv2.ROTATE_90_CLOCKWISE))
plt.subplot(1,4,3), plt.imshow(cv2.rotate(image.copy(), cv2.ROTATE_90_COUNTERCLOCKWISE))
plt.subplot(1,4,4), plt.imshow(cv2.rotate(image.copy(), cv2.ROTATE_180))

for ax in plt.gcf().axes:
    ax.axis("off")

print(f"ROTATE_90_CLOCKWISE = {cv2.ROTATE_90_CLOCKWISE}")
print(f"ROTATE_90_COUNTERCLOCKWISE = {cv2.ROTATE_90_COUNTERCLOCKWISE}")
print(f"ROTATE_180 = {cv2.ROTATE_180}")

In [ ]:
plt.figure(figsize=(15,6))

plt.subplot(1,4,1), plt.imshow(image)
plt.subplot(1,4,2), plt.imshow(cv2.flip(image.copy(), 1))
plt.subplot(1,4,3), plt.imshow(cv2.flip(image.copy(), 0))
plt.subplot(1,4,4), plt.imshow(cv2.flip(image.copy(), -1))

for ax in plt.gcf().axes:
    ax.axis("off")

이미지는 기본적으로 픽셀 값을 [0,255]으로 저장하며, 데이터 타입은 `uint8`입니다.

In [ ]:
image.dtype

데이터 타입을 [0,255] 8비트 정수에서 [0,1] 32비트 실수로 변환해봅시다.

In [ ]:
plt.imshow(image.astype(np.float32))
plt.axis("off");

Matplotlib이 실수 값은 [0,1]인데 왜 그 밖의 숫자를 건네주는지 불평하는군요.

각 픽셀 값을 255로 나눠줍시다.

In [ ]:
plt.imshow(image.astype(np.float32) / 255.)
plt.axis("off");

이제 32비트 실수를 사용하여 이전과 같은 이미지를 볼 수 있습니다.

### **B. 색공간 변환**

색공간 (Color Space)이란?
* 색을 일정한 기준으로 표현하는 방식
* 색공간마다 표현 가능한 색의 범위가 다름
* 일관된 색 표현을 위해 색공간 통일
* 색공간 변환을 통해 이미지 전처리 효율을 극대화

#### $1)$ RGB와 BGR 색공간

기본적으로 Matplotlib은 RGB, OpenCV는 BGR 색공간을 사용합니다.

이번에는 OpenCV로 이미지를 불러옵시다.

In [ ]:
import requests

img_url = "https://upload.wikimedia.org/wikipedia/commons/3/30/Girl_with_a_Pearl_Earring_%28c._1665%29.webp?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=original"
img_opencv = cv2.imdecode(np.frombuffer(requests.get(img_url, headers={"User-Agent": "Mozilla/5.0"}).content, np.uint8), cv2.IMREAD_COLOR)

plt.imshow(img_opencv)
plt.axis("off");

무언가 확실히 잘못됐죠?

그 이유는 OpenCV는 기본적으로 BGR 순서로 이미지를 읽어오지만, Matplotlib은 RGB 순서 기준으로 이미지를 해석하기 때문입니다.

따라서 최종적으로 이미지를 렌더링할 시각화 도구의 색공간 기준으로 변환해 주어야 합니다.

OpenCV의 `cv2.cvtColor` 함수를 사용하여 변환해봅시다.

In [ ]:
img_rgb = cv2.cvtColor(img_opencv, cv2.COLOR_BGR2RGB)

plt.imshow(img_rgb)
plt.axis("off");

#### $2)$ Grayscale

동일한 방식으로 흑백 (grayscale)로 변환할 수 있습니다.

In [ ]:
img_gray = cv2.cvtColor(img_opencv, cv2.COLOR_BGR2GRAY)

plt.imshow(img_gray, cmap="gray")
plt.axis("off");

#### $3)$ HSV 색공간

HSV 색공간:
* H (Hue): 색상
  * 일반: $0^\circ$ ~ $360^\circ$
  * OpenCV: $0^\circ$ ~ $179^\circ$ (uint8), $0.0^\circ$ ~ $360.0^\circ$ (float32)
* S (Saturation): 채도
  * 일반: 0 ~ 100%
  * OpenCV: 0 ~ 255 (uint8), 0.0 ~ 1.0 (float32)
* V (Value): 명도
  * 일반: 0 ~ 100%
  * OpenCV: 0 ~ 255 (uint8), 0.0 ~ 1.0 (float32)

이제 HSV 색공간으로 변환하여 각 channel을 시각화해 봅시다.

`cv2.split(이미지)` 함수를 사용하면 이미지의 각 channel을 분리할 수 있습니다.

In [ ]:
image_hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
h, s, v = cv2.split(image_hsv)

plt.figure(figsize=(15,8))

plt.subplot(1,3,1), plt.imshow(h, cmap="gray"), plt.title('H Channel (Hue)')
plt.subplot(1,3,2), plt.imshow(s, cmap="gray"), plt.title('S Channel (Saturation)')
plt.subplot(1,3,3), plt.imshow(v, cmap="gray"), plt.title('V Channel (Value)')

for ax in plt.gcf().axes:
    ax.axis("off")

HSV 각 채널을 분리하였다면 각 색상/채도/명도를 조절하여 이미지를 수정할 수 있습니다.

H (색상) 각도를 shift하여 색상을 변경해 봅시다.

H의 범위가 (float32 기준) $0.0^\circ$ ~ $360.0^\circ$이기에 360도가 넘어가면 0으로 순환하도록 코드를 작성합시다.

In [ ]:
image = skimage.data.astronaut()
image = image.astype(np.float32) / 255.

image_hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
h, s, v = cv2.split(image_hsv)

h = (h + 45) % 360
image_hsv = cv2.merge((h, s, v))
image_rgb = cv2.cvtColor(image_hsv, cv2.COLOR_HSV2RGB)

plt.figure(figsize=(15,8))
plt.subplot(1,3,1), plt.imshow(image), plt.title('Original')
plt.subplot(1,3,2), plt.imshow(image_rgb), plt.title('H Value Modified')

for ax in plt.gcf().axes:
    ax.axis("off")

S (채도) 값을 증폭시켜 채도를 높여봅시다.

S의 범위가 (float32 기준) 0.0 ~ 1.0 이기에 범위에서 벗어나지 않도록 코드를 작성합시다.

In [ ]:
image = skimage.data.astronaut()
image = image.astype(np.float32) / 255.

image_hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
h, s, v = cv2.split(image_hsv)

s = np.clip(s * 1.2, 0, 1)
image_hsv = cv2.merge((h, s, v))
image_rgb = cv2.cvtColor(image_hsv, cv2.COLOR_HSV2RGB)

plt.figure(figsize=(15,8))
plt.subplot(1,3,1), plt.imshow(image), plt.title('Original')
plt.subplot(1,3,2), plt.imshow(image_rgb), plt.title('S Value Modified')

for ax in plt.gcf().axes:
    ax.axis("off")

V (명도) 값을 감쇠시켜 명도를 낮춰봅시다.

V의 범위가 (float32 기준) 0.0 ~ 1.0 이기에 범위에서 벗어나지 않도록 코드를 작성합시다.

In [ ]:
image = skimage.data.astronaut()
image = image.astype(np.float32) / 255.

image_hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
h, s, v = cv2.split(image_hsv)

v = np.clip(v * 0.3, 0, 1)
image_hsv = cv2.merge((h, s, v))
image_rgb = cv2.cvtColor(image_hsv, cv2.COLOR_HSV2RGB)

plt.figure(figsize=(15,8))
plt.subplot(1,3,1), plt.imshow(image), plt.title('Original')
plt.subplot(1,3,2), plt.imshow(image_rgb), plt.title('S Value Modified')

for ax in plt.gcf().axes:
    ax.axis("off")

앞선 실습에서 확인했듯이, HSV 색공간은 색상/채도/명도가 독립적으로 분리되어 있습니다.

따라서 각 채널을 개별적으로 조절하여 이미지를 원하는 대로 변형할 수 있습니다.

### **C. 객체 탐지**

#### $1)$ Color Segmentation

색상 기반 분할 (Color Segmentation)이란?
* 기법 :   색상 기반 분할 (Color Segmentation)
* 설명 :   원하는 특정 색상만 분리/추출
* 방법 :   색상 임계값 처리  (Color Thresholding)
* 결과 :   이진 마스크  (Binary Mask)

이미지 내 빨간색 Color Thresholding을 통해 이진 마스크(Binary Mask)를 생성해봅시다.

In [ ]:
motor = skimage.data.stereo_motorcycle()[0]

plt.imshow(motor)
plt.axis("off");

우선 Color Thresholding을 위해 이미지를 HSV 색공간으로 변환해줍시다.

그 다음, `cv2.inRange(hsv_이미지, color_lower, color_upper)` 함수를 사용하여 mask를 생성합니다.

빨강은 $0^\circ$에 가깝기에 H값 $0^\circ$ ~ ($0$ + $\theta)^\circ$와 $(180-\theta)^\circ$ ~ $180^\circ$ 두 가지 mask를 bitwise OR로 연산합니다.

In [ ]:
red_lower1 = np.array([0, 100, 50])
red_upper1 = np.array([10, 255, 255])
red_lower2 = np.array([170, 100, 50])
red_upper2 = np.array([180, 255, 255])

motor_hsv = cv2.cvtColor(motor, cv2.COLOR_RGB2HSV)

mask_hsv1 = cv2.inRange(motor_hsv, red_lower1, red_upper1)
mask_hsv2 = cv2.inRange(motor_hsv, red_lower2, red_upper2)
mask_hsv = cv2.bitwise_or(mask_hsv1, mask_hsv2)

plt.figure(figsize=(16,8))

plt.subplot(1,2,1), plt.imshow(motor), plt.title("Original")
plt.subplot(1,2,2), plt.imshow(mask_hsv, cmap="gray"), plt.title("Binary Mask")

for ax in plt.gcf().axes:
    ax.axis("off")

Binary Mask에 노이즈가 너무 많습니다.

노이즈를 제거하기 위해 Low-Pass Filter (blur)를 적용합시다.

In [ ]:
motor_blur = cv2.GaussianBlur(motor, (7, 7), 0)

motor_blur_hsv = cv2.cvtColor(motor_blur, cv2.COLOR_RGB2HSV)

mask_blur_hsv1 = cv2.inRange(motor_blur_hsv, red_lower1, red_upper1)
mask_blur_hsv2 = cv2.inRange(motor_blur_hsv, red_lower2, red_upper2)
mask_blur_hsv = cv2.bitwise_or(mask_blur_hsv1, mask_blur_hsv2)

plt.figure(figsize=(16,8))

plt.subplot(1,2,1), plt.imshow(mask_hsv, cmap="gray"), plt.title("Binary Mask")
plt.subplot(1,2,2), plt.imshow(mask_blur_hsv, cmap="gray"), plt.title("Binary Mask (Blurred)")

for ax in plt.gcf().axes:
    ax.axis("off")

너무 강하게 Smoothing하면 정보 유실이 크기에 약하게만 Smoothing Filter를 적용합니다.

#### $2)$ Morphological Operation

그럼 남아있는 노이즈는 어떻게 처리할까요?

모폴로지 연산 (Morphological Operation)이란?
* 형태학적 연산
* Mask의 영역을 확대/축소
* Erosion(침식), Dilation(팽창), Opening(열기), Closing(닫기)

Erosion (침식)  vs.  Dilation (팽창):
* Erosion (침식)
  * Kernel 내 모든 픽셀이 1 (흰색)일 때만 중심 픽셀을 1 (흰색)으로 설정
  * AND
  * 흰색 영역 감소
  * 배경의 노이즈 제거
* Dilation (팽창)
  * Kernel 내 픽셀 하나라도 1 (흰색)이면 중심 픽셀을 1 (흰색)으로 설정
  * OR
  * 흰색 영역 확장
  * 내부의 검정 구멍과 내부의 끊어진 윤곽선 연결

In [ ]:
# kernel = np.ones((3, 3))
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

mask_blur_hsv_opened = cv2.morphologyEx(mask_blur_hsv, cv2.MORPH_OPEN, kernel)

plt.figure(figsize=(16,8))

plt.subplot(1,3,1), plt.imshow(mask_hsv, cmap="gray"), plt.title("Binary Mask")
plt.subplot(1,3,2), plt.imshow(mask_blur_hsv, cmap="gray"), plt.title("Binary Mask (Blurred)")
plt.subplot(1,3,3), plt.imshow(mask_blur_hsv_opened, cmap="gray"), plt.title("Binary Mask (Opened)")

for ax in plt.gcf().axes:
    ax.axis("off")

#### $3)$ Contour Detection

외곽선 (Contour)이란?
* binary mask에서 연결된 영역의 외곽선
* 연결된 영역이기에 적절한 morphological operation 중요

오토바이 이미지로 한번 contour detection을 시도해봅시다.

우선 이미지에 약한 blur를 적용한 뒤 binary mask를 생성합니다.

In [ ]:
# Gaussin blur 적용
motor_blur = cv2.GaussianBlur(motor, (7, 7), 0)

# HSV로 변환
motor_blur_hsv = cv2.cvtColor(motor_blur, cv2.COLOR_RGB2HSV)

# 빨간색 추출하여 binary mask 생성
motor_mask1 = cv2.inRange(motor_blur_hsv, red_lower1, red_upper1)
motor_mask2 = cv2.inRange(motor_blur_hsv, red_lower2, red_upper2)
motor_mask = cv2.bitwise_or(motor_mask1, motor_mask2)

plt.figure(figsize=(16,8))

plt.subplot(1,2,1), plt.imshow(motor), plt.title("Original Image")
plt.subplot(1,2,2), plt.imshow(motor_mask, cmap="gray"), plt.title("Binary Mask (Blurred)")

for ax in plt.gcf().axes:
    ax.axis("off")

그 다음, morphological operation으로 mask를 한층 더 깔끔하게 수정합니다.

Opening 2회, Closing 3회를 적용하여 최종 binary mask를 생성하였습니다.

In [ ]:
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

motor_mask_opened2 = cv2.morphologyEx(motor_mask, cv2.MORPH_OPEN, kernel, iterations=2)
motor_mask_closed3 = cv2.morphologyEx(motor_mask_opened2, cv2.MORPH_CLOSE, kernel, iterations=3)

plt.figure(figsize=(16,8))
plt.subplot(1,3,1), plt.imshow(motor, cmap="gray"), plt.title("Original Image")
plt.subplot(1,3,2), plt.imshow(motor_mask, cmap="gray"), plt.title("Binary Mask (Blurred)")
plt.subplot(1,3,3), plt.imshow(motor_mask_closed3, cmap="gray"), plt.title("Opening x2 & Closing x3")

for ax in plt.gcf().axes:
    ax.axis("off")

자, 이제 binary mask 영역의 contour를 추출해봅시다.

In [ ]:
# 최종 binary mask
final_motor_mask = motor_mask_closed3

# Contour를 그릴 원본 이미지 copy
final_img = motor.copy()

# 컨투어 추출 (두 번째 반환값인 hierarchy는 바깥쪽 테두리만 검출하는 현재 방식상 무의미하기에 저장하지 않음)
contours, _ = cv2.findContours(
    final_motor_mask,        # binary mask
    cv2.RETR_EXTERNAL,       # 구멍 뚫린 객체의 바깥쪽 테두리만 검출
    cv2.CHAIN_APPROX_SIMPLE  # 직선 경로 위에 있는 픽셀 좌표 전체 대신 꼭짓점 좌표만 저장
)

if len(contours) > 0:
    # Binary mask를 반투명 초록색으로 overlay
    mask_color = np.zeros_like(final_img)
    mask_color[:, :, 1] = final_motor_mask
    alpha = 0.35
    final_img = cv2.addWeighted(final_img, 1.0, mask_color, alpha, 0)

    # Contour 그리기 (초록색 외곽선)
    cv2.drawContours(final_img, contours, -1, (0,255,0), 2)

plt.imshow(final_img)
plt.axis("off");

이진 마스크(Binary Mask) 및 윤곽선 검출(Contour Detection)을 수행하여 이미지의 빨간색 영역(반투명 초록색)과 해당 외곽선(진한 초록색)을 추출하였습니다.

하지만 저희가 원하는건 오직 오토바이의 빨간 영역이죠.

그러기 위해서는 검출된 윤곽선 중 면적이 가장 크거나 둘레가 가장 긴 외곽선을 최종 선택합니다.

일반적으로 노이즈 및 불필요한 영역은 상대적으로 작은 면적을 형성하기 때문입니다.

이전 단계에서 Morphological Operation을 신중히 적용한 목적 역시 주 영역의 연속성을 확보하여 윤곽선 면적을 최대화하기 위함이었습니다.

In [ ]:
# 최종 binary mask
final_motor_mask = motor_mask_closed3

# Contour를 그릴 원본 이미지 copy
final_img = motor.copy()

# 컨투어 추출 (두 번째 반환값인 hierarchy는 바깥쪽 테두리만 검출하는 현재 방식상 무의미하기에 저장하지 않음)
contours, _ = cv2.findContours(
    final_motor_mask,        # binary mask
    cv2.RETR_EXTERNAL,       # 구멍 뚫린 객체의 바깥쪽 테두리만 검출
    cv2.CHAIN_APPROX_SIMPLE  # 직선 경로 위에 있는 픽셀 좌표 전체 대신 꼭짓점 좌표만 저장
)

if len(contours) > 0:
    # 가장 큰 contour
    largest_contour = max(contours, key=cv2.contourArea)

    # 가장 큰 contour만 채운 binary mask 생성
    final_mask = np.zeros_like(final_motor_mask)
    cv2.drawContours(
        final_mask,
        [largest_contour],
        -1,
        255,
        thickness=cv2.FILLED
    )

    # 가장 큰 contour만 채운 binary mask를 반투명 초록색으로 overlay
    mask_color = np.zeros_like(final_img)
    mask_color[:, :, 1] = final_mask
    alpha = 0.35
    final_img = cv2.addWeighted(final_img, 1.0, mask_color, alpha, 0)

    # Contour 그리기 (초록색 외곽선)
    cv2.drawContours(final_img, [largest_contour], -1, (0,255,0), 2)

plt.imshow(final_img)
plt.axis("off");

#### $4)$ Minimum Enclosing Circle

이제 객체의 중심 및 반지름을 찾아 그려봅시다.

최소 외접원 (Minimum Enclosing Circle):

“외곽선의 모든 점들을 포함하는 가장 작은 원”

In [ ]:
# 최종 binary mask
final_ball_mask = motor_mask_closed3

# Contour 및 외접원을 그릴 원본 이미지 copy
final_img = motor.copy()

# 컨투어 추출 (두 번째 반환값인 hierarchy는 바깥쪽 테두리만 검출하는 현재 방식상 무의미하기에 저장하지 않음)
contours, _ = cv2.findContours(
    final_motor_mask,        # binary mask
    cv2.RETR_EXTERNAL,       # 구멍 뚫린 객체의 바깥쪽 테두리만 검출
    cv2.CHAIN_APPROX_SIMPLE  # 직선 경로 위에 있는 픽셀 좌표 전체 대신 꼭짓점 좌표만 저장
)

if len(contours) > 0:
    # 가장 큰 contour
    largest_contour = max(contours, key=cv2.contourArea)

    # 가장 큰 contour만 채운 binary mask 생성
    largest_mask = np.zeros_like(final_ball_mask)
    cv2.drawContours(
        largest_mask,
        [largest_contour],
        -1,
        255,
        thickness=cv2.FILLED
    )

    # 가장 큰 contour만 채운 binary mask를 반투명 초록색으로 overlay
    mask_color = np.zeros_like(final_img)
    mask_color[:, :, 1] = final_mask
    alpha = 0.35
    final_img = cv2.addWeighted(final_img, 1.0, mask_color, alpha, 0)

    # Contour 그리기 (초록색 외곽선)
    cv2.drawContours(final_img, [largest_contour], -1, (0,255,0), 2)

    # 최소 외접원 중심 및 반지름
    (x, y), radius = cv2.minEnclosingCircle(largest_contour)
    center_circle = (int(x), int(y))
    radius = int(radius)

    # 파란색 원
    cv2.circle(final_img, center_circle, radius, (0,0,255), 3)
    cv2.circle(final_img, center_circle, 5, (0,0,255), -1)

plt.imshow(final_img)
plt.axis("off");

#### $5)$ Centroid

무게중심(Centroid):

"객체/Contour의 픽셀 분포를 기반으로 계산한 기하학적 중심점(객체의 중심 좌표)"

무게중심  =  픽셀 가중치 합 / 전체 면적
$$
C_x = \frac{M_{10}}{M_{00}}, \qquad
C_y = \frac{M_{01}}{M_{00}}
$$

In [ ]:
# 최종 binary mask
final_ball_mask = motor_mask_closed3

# Contour 및 외접원을 그릴 원본 이미지 copy
final_img = motor.copy()

# 컨투어 추출 (두 번째 반환값인 hierarchy는 바깥쪽 테두리만 검출하는 현재 방식상 무의미하기에 저장하지 않음)
contours, _ = cv2.findContours(
    final_motor_mask,        # binary mask
    cv2.RETR_EXTERNAL,       # 구멍 뚫린 객체의 바깥쪽 테두리만 검출
    cv2.CHAIN_APPROX_SIMPLE  # 직선 경로 위에 있는 픽셀 좌표 전체 대신 꼭짓점 좌표만 저장
)

if len(contours) > 0:
    # 가장 큰 contour
    largest_contour = max(contours, key=cv2.contourArea)

    # 가장 큰 contour만 채운 binary mask 생성
    largest_mask = np.zeros_like(final_ball_mask)
    cv2.drawContours(
        largest_mask,
        [largest_contour],
        -1,
        255,
        thickness=cv2.FILLED
    )

    # 가장 큰 contour만 채운 binary mask를 반투명 초록색으로 overlay
    mask_color = np.zeros_like(final_img)
    mask_color[:, :, 1] = final_mask
    alpha = 0.35
    final_img = cv2.addWeighted(final_img, 1.0, mask_color, alpha, 0)

    # Contour 그리기 (초록색 외곽선)
    cv2.drawContours(final_img, [largest_contour], -1, (0,255,0), 2)

    # Moments를 통해 무게중심 (Centroid) 산출
    M = cv2.moments(largest_contour)

    if M["m00"] != 0:
        cX = int(M["m10"] / M["m00"])
        cY = int(M["m01"] / M["m00"])
    else:
        cX, cY = 0, 0

    centroid = (cX, cY)

    # 분홍색 무게중심
    cv2.circle(final_img, centroid, 15, (255,0,255), -1)

plt.imshow(final_img)
plt.axis("off");

무게중심을 산출하는 과정에서 반지름은 구할 수 없기에 최소 외접원 반지름과 함께 혼합하여 최종적인 공 영역을 예측할 수 있습니다.

In [ ]:
# 최종 binary mask
final_ball_mask = motor_mask_closed3

# Contour 및 외접원을 그릴 원본 이미지 copy
final_img = motor.copy()

# 컨투어 추출 (두 번째 반환값인 hierarchy는 바깥쪽 테두리만 검출하는 현재 방식상 무의미하기에 저장하지 않음)
contours, _ = cv2.findContours(
    final_motor_mask,        # binary mask
    cv2.RETR_EXTERNAL,       # 구멍 뚫린 객체의 바깥쪽 테두리만 검출
    cv2.CHAIN_APPROX_SIMPLE  # 직선 경로 위에 있는 픽셀 좌표 전체 대신 꼭짓점 좌표만 저장
)

if len(contours) > 0:
    # 가장 큰 contour
    largest_contour = max(contours, key=cv2.contourArea)

    # 가장 큰 contour만 채운 binary mask 생성
    largest_mask = np.zeros_like(final_ball_mask)
    cv2.drawContours(
        largest_mask,
        [largest_contour],
        -1,
        255,
        thickness=cv2.FILLED
    )

    # 가장 큰 contour만 채운 binary mask를 반투명 초록색으로 overlay
    mask_color = np.zeros_like(final_img)
    mask_color[:, :, 1] = final_mask
    alpha = 0.35
    final_img = cv2.addWeighted(final_img, 1.0, mask_color, alpha, 0)

    # Contour 그리기 (초록색 외곽선)
    cv2.drawContours(final_img, [largest_contour], -1, (0,255,0), 2)

    # 최소 외접원 중심 및 반지름
    (x, y), radius = cv2.minEnclosingCircle(largest_contour)
    center_circle = (int(x), int(y))
    radius = int(radius)

    # Moments를 통해 무게중심 (Centroid) 산출
    M = cv2.moments(largest_contour)

    if M["m00"] != 0:
        cX = int(M["m10"] / M["m00"])
        cY = int(M["m01"] / M["m00"])
    else:
        cX, cY = 0, 0

    centroid = (cX, cY)

    # 분홍색 무게중심 기반 원
    cv2.circle(final_img, centroid, radius, (255,0,255), 3)
    cv2.circle(final_img, centroid, 5, (255,0,255), -1)

plt.imshow(final_img)
plt.axis("off");

최소 외접원 vs. 무게중심 최종 비교

In [ ]:
# 최종 binary mask
final_ball_mask = motor_mask_closed3

# Contour 및 외접원을 그릴 원본 이미지 copy
final_img = motor.copy()

# 컨투어 추출 (두 번째 반환값인 hierarchy는 바깥쪽 테두리만 검출하는 현재 방식상 무의미하기에 저장하지 않음)
contours, _ = cv2.findContours(
    final_motor_mask,        # binary mask
    cv2.RETR_EXTERNAL,       # 구멍 뚫린 객체의 바깥쪽 테두리만 검출
    cv2.CHAIN_APPROX_SIMPLE  # 직선 경로 위에 있는 픽셀 좌표 전체 대신 꼭짓점 좌표만 저장
)

if len(contours) > 0:
    # 가장 큰 contour
    largest_contour = max(contours, key=cv2.contourArea)

    # 가장 큰 contour만 채운 binary mask 생성
    largest_mask = np.zeros_like(final_ball_mask)
    cv2.drawContours(
        largest_mask,
        [largest_contour],
        -1,
        255,
        thickness=cv2.FILLED
    )

    # 가장 큰 contour만 채운 binary mask를 반투명 초록색으로 overlay
    mask_color = np.zeros_like(final_img)
    mask_color[:, :, 1] = final_mask
    alpha = 0.35
    final_img = cv2.addWeighted(final_img, 1.0, mask_color, alpha, 0)

    # Contour 그리기 (초록색 외곽선)
    cv2.drawContours(final_img, [largest_contour], -1, (0,255,0), 2)

    # 최소 외접원 중심 및 반지름
    (x, y), radius = cv2.minEnclosingCircle(largest_contour)
    center_circle = (int(x), int(y))
    radius = int(radius)

    # Moments를 통해 무게중심 (Centroid) 산출
    M = cv2.moments(largest_contour)

    if M["m00"] != 0:
        cX = int(M["m10"] / M["m00"])
        cY = int(M["m01"] / M["m00"])
    else:
        cX, cY = 0, 0

    centroid = (cX, cY)

    # 파란색 최소 외접원
    cv2.circle(final_img, center_circle, radius, (0,0,255), 3)
    cv2.circle(final_img, center_circle, 5, (0,0,255), -1)

    # 분홍색 무게중심 기반 원
    cv2.circle(final_img, centroid, radius, (255,0,255), 3)
    cv2.circle(final_img, centroid, 5, (255,0,255), -1)

plt.imshow(final_img)
plt.axis("off");

### **D. 학습 기반 객체 탐지**

**YOLO (You Only Look Once):**
* CNN 기반 객체 탐지 모델
* One-stage Detector
* 이미지 전체를 신경망에 한 번 입력하여, 이미지 속 객체들을 전부 탐지
* 객체들의 정보 동시 예측
  * 위치: Bounding Box
  * 종류: Class
  * 신뢰도: Confidence
* 실시간 처리에 적합
  * 별도의 Region Proposal 단계가 없음
  * Feature Map을 여러 객체가 공유함
  * 여러 위치의 예측을 병렬로 계산함
  * GPU의 병렬 연산을 효율적으로 활용
  * 작은 모델부터 큰 모델까지 선택 가능

우선, YOLO 모델을 활용하기 위해 공식 파이썬 라이브러리인 `ultralytics`를 설치해야합니다.

`ultralytics` 패키지를 설치하기 위해서는 딥러닝 프레임워크인 `PyTorch`를 먼저 설치해야 합니다.

`PyTorch`:
* GPU 연산을 지원하는 오픈소스 딥러닝 프레임워크
* NumPy 배열과 유사한 다차원 배열인 Tensor 사용
* NVIDIA CUDA를 활용하여 딥러닝 연산을 GPU에서 가속
* 자동 미분을 통해 신경망의 가중치와 편향을 학습
* CNN, RNN, Transformer 등 다양한 딥러닝 모델 구현 가능

<br>

`PyTorch` 활용 사례:
* 이미지 분류
* 객체 검출
* 이미지 Segmentation
* 자세 추정
* 자연어 처리
* 음성 인식
* 생성형 AI
* 강화학습
* 로봇 인지
* Edge AI 추론

<br>

`PyTorch` 주요 기능:
* Tensor 연산
* 자동 미분
* 신경망 Layer, 손실 함수, Optimizer 제공
* 데이터 로딩 및 전처리
* 모델 구성 및 학습
* 모델 저장 및 불러오기

PyTorch는 로봇의 카메라 이미지와 같은 데이터를 이용하여 딥러닝 모델을 실행하는 역할을 합니다. 이때 모델의 연산을 CPU에서 수행할 것인지, GPU에서 수행할 것인지에 따라 설치 방법이 달라질 수 있습니다.

특히 NVIDIA GPU를 사용하는 경우에는 GPU의 CUDA 환경에 맞는 PyTorch 패키지를 설치해야 합니다. 따라서 자신의 GPU 및 CUDA 환경을 확인한 후 적절한 설치 명령어를 선택합니다.

```bash
pip install torch==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cpu
pip install torch==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/<CUDA_version>
pip install torch==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/<GPU_backend>
```

정상적으로 설치가 되었는지 아래 코드로 확인해봅시다.

In [ ]:
from ultralytics import YOLO

에러 없이 `ultralytics` 라이브러리가 import 되었다면 다음으로 진행하면 되겠습니다.

#### $1)$ YOLO 모델 선택

YOLO는 사용 목적과 컴퓨터 환경에 맞게 선택할 수 있도록 매우 다양한 모델 라인업을 제공합니다.

YOLO 모델의 이름은 크게 ***모델명 + 버전 + 모델 크기***의 구조로 이루어져 있습니다.

예) YOLO11n
* **YOLO**: **모델 이름** *(You Only Look Once)*
* **숫자 (11)**: **버전 정보** (V1부터 지속적으로 발전하여 최근에는 **YOLO11**까지 출시)
* **알파벳 (n)**: **모델 크기 및 파라미터 수** (n, s, m, l, x 등)

모델 크기(알파벳) 구분

| 알파벳 | 약자 | 특징 |
| :---: | :---: | :--- |
| **n** | Nano | 경량화 모델 (속도 최우선, 연산량 적음) |
| **s** | Small | 소형 모델 (가벼우면서 적절한 성능) |
| **m** | Medium | 중형 모델 (속도와 정확도의 균형) |
| **l** | Large | 대형 모델 (높은 정확도) |
| **x** | Extra Large | 초대형 모델 (최고 성능, 높은 연산량 필요) |

YOLO는 최신 버전인 `YOLO26`까지 출시되어 있지만, 본 수업에서는 아래와 같은 이유로 `YOLO11n` 모델을 선택하여 진행합니다.

1. **안정성과 호환성**: 최신 `YOLO26`에 비해 `YOLO11`은 라이브러리 및 실행 환경(PyTorch, OpenCV 등)과의 의존성 충돌 위험이 적고 매우 안정적

2. **실시간 가벼운 동작**: `n(Nano)` 모델은 연산량이 적어 노트북, PC, 엣지 디바이스 등 다양한 실습 환경에서 실시간 카메라 피드 탐지를 무리 없이 수행

3. **빠르고 효율적인 실습**: 모델 가중치 파일 용량이 가벼워 로딩이 빠르며, 경량화 환경에서도 YOLO의 핵심 메커니즘을 실행하기에 최적화된 모델

#### $2)$ COCO 데이터셋

COCO 데이터셋:
* **컴퓨터 비전 대표 데이터셋**
  * Microsoft에서 제작한 대규모 객체 탐지(Object Detection) 및 분할(Segmentation) 데이터셋
* **80가지 클래스 제공**
  * 사람, 자동차, 강아지, 의자 등 일상에서 흔히 접하는 80가지 범주의 객체 정보가 포함되어 높은 활용도
* **복잡한 배경과 맥락(Context)**
  * 단일 물체만 깔끔하게 있는 이미지뿐만 아니라, 여러 객체가 어우러진 실제 환경 이미지가 다수 포함되어 있어 높은 실전 인식 성능

#### $3)$ YOLO 모델 최초 설치 및 불러오기

이제 `YOLO11n` 모델을 불러와 보겠습니다.

파일을 지정한 경로에서 불러오거나 파일이 존재하지 않는다면 자동으로 다운로드됩니다.

최초 사용하는 모델이므로, 경로를 지정하여 설치해봅시다.

In [ ]:
model = YOLO("models/YOLO/yolo11n.pt")

Ultralytics는 GPU가 사용 가능한 환경이라면 아래와 같이 모델을 GPU에 할당 할 수 있습니다.

```python
model.to("cuda")   # NVIDIA GPU
model.to("mps")    # Apple Silicon GPU
model.to("xpu")    # Intel GPU
model.to("cpu")    # CPU
```

#### $4)$ 정적 이미지에서의 YOLO 객체 탐지

이제 불러온 모델에 샘플 이미지를 적용하여 객체 탐지가 잘 동작하는지 확인해보겠습니다.

In [ ]:
results = model.predict(
    source="https://upload.wikimedia.org/wikipedia/commons/8/88/Traffic_at_Harvard_Square%2C_Cambridge%2C_Mass._-_DPLA_-_557905360431b653bac383722b9c511e.jpg?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=original",
    conf=0.25,   # Confidence Threshold
    iou=0.5,     # IoU Threshold
    classes=None,
)

yolo_test_img = results[0].plot()

yolo_test_img_rgb = cv2.cvtColor(yolo_test_img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12,8))
plt.imshow(yolo_test_img_rgb)
plt.axis("off");

#### $5)$ 클래스 지정 객체 탐지

YOLO 예측 함수인 `model.predict()`에는 원하는 클래스를 지정하여 객체를 탐지할 수 있는 기능이 있습니다.

In [ ]:
results = model.predict(
    source="https://upload.wikimedia.org/wikipedia/commons/8/88/Traffic_at_Harvard_Square%2C_Cambridge%2C_Mass._-_DPLA_-_557905360431b653bac383722b9c511e.jpg?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=original",
    conf=0.25,   # Confidence Threshold
    iou=0.5,     # IoU Threshold
    classes=[1, 26],
)

yolo_test_img = results[0].plot()

yolo_test_img_rgb = cv2.cvtColor(yolo_test_img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12,8))
plt.imshow(yolo_test_img_rgb)
plt.axis("off");

---

## 3. Planning

AMR이 주변 환경을 인식하고 자신의 위치를 추정했다면, 다음 단계는 어디로 이동할지, 그리고 어떻게 이동할지 결정하는 것입니다.

로봇이 현재 위치에서 목표 위치까지 안전하고 효율적으로 이동하기 위해 계획하는 과정을 Planning 이라고 합니다.

### **A. Global Planning과 Local Planning**

**Global Planner**는 로봇이 이동해야 할 전체적인 경로를 계획합니다. SLAM을 통해 생성된 지도에서 로봇의 현재 위치와 목표 위치를 알고 있다면, 지도에 존재하는 장애물을 고려하여 목표까지의 경로를 계산할 수 있습니다. 대표적인 경로 탐색 알고리즘으로는 **Dijkstra**와 **A***가 있습니다.

반면 **Local Planner**는 로봇이 실제로 이동하는 과정에서 주변 상황에 대응합니다. 예를 들어 계획된 경로를 따라 이동하던 중 사람이 갑자기 로봇 앞을 지나간다면, 로봇은 사람과 충돌하지 않도록 일시적으로 속도를 줄이거나 이동 방향을 변경해야 합니다.

* Global Planner: 시작점에서 목표점까지 가기 위해 경로를 계산하기 위한 시스템/모듈

* Global Path: Global Planner가 계산한 시작점에서 목표점까지의 전체적인 이동 경로

* Local Planner/Controller: 현재 위치와 주변 환경을 고려하여 가까운 구간에서 움직임을 계산·제어하는 시스템/모듈

* Local Path: Local Planner/Controller가 현재 움직임을 제어하기 위해 사용하는 가까운 구간의 경로


### **B. Dijkstra와 A***

Global Planner의 대표적인 경로 탐색 알고리즘인 **Dijkstra**와 **A***를 알아보도록 하겠습니다.

두 알고리즘 모두 그래프(Graph)에서 시작 지점부터 목표 지점까지의 경로를 탐색하는 알고리즘입니다. AMR에서는 지도 공간을 여러 개의 작은 영역으로 나누고, 각 영역을 하나의 Node로 표현할 수 있습니다. 인접한 Node 사이에는 로봇이 이동할 수 있는 연결 관계인 Edge가 존재합니다.

이러한 방식으로 로봇이 이동할 수 있는 공간을 그래프로 표현하면, 경로 탐색 알고리즘을 이용하여 시작 위치에서 목표 위치까지 이동할 경로를 계산할 수 있습니다.

#### $1)$ **Dijkstra**

**Dijkstra** 알고리즘은 시작 지점에서 다른 모든 위치까지의 최단 경로를 탐색하는 알고리즘입니다.

현재까지 계산된 이동 비용이 가장 작은 위치부터 차례대로 탐색하면서, 각 위치까지의 최단 거리를 업데이트합니다. 목표 위치까지의 예상 비용을 별도로 사용하지 않고, 현재까지 실제로 이동한 비용만을 기준으로 다음 탐색 위치를 결정합니다.

따라서 최단 경로를 찾을 수 있다는 장점이 있지만, 목표와 관계없이 넓은 영역을 탐색할 수 있기 때문에 탐색해야 하는 Node가 많아질 수 있습니다.

##### Dijkstra

* 출발지부터 모든 격자를 탐색  →  최소 비용의 경로 계산

* 목표 노드까지 모든 grid에 비용 점수 부여

* BFS (Breadth-First Search)

* $f(n) = g(n)$

* 장점:  무조건 최적화된 해답 찾음

* 단점:  연산량 메모리

#### $2)$ **A***

**A*** 알고리즘은 Dijkstra를 기반으로 하면서 목표 위치까지의 예상 비용을 함께 사용하는 알고리즘입니다. 지금까지 얼마나 이동했는지 뿐만 아니라 목표 지점까지 얼마나 가까운지도 고려하여 탐색할 위치를 결정합니다.

**A***에서는 각 Node의 우선순위를 다음과 같이 계산합니다.

$$
f(n) = g(n) + h(n)
$$

여기서 $h(n)$을 계산하는 방법을 Heuristic Function 이라고 합니다.

격자 형태의 지도에서는 대표적으로 다음과 같은 방법을 사용할 수 있습니다.

**Manhattan Distance:**

$$
h(n) = |x_b - x_a| + |y_b - y_a|
$$

**Euclidean Distance:**

$$
h(n) = \sqrt{(x_b - x_a)^2 + (y_b - y_a)^2}
$$

##### A*

* Dijkstra + Heuristics

* 해당 노드까지 비용 + 목적지까지의 남은 거리

* 목적지 방향 위주로 우선 탐색

* $f(n) = g(n) + h(n)$

* $g(n)$: 시작 지점에서 현재 Node n까지 실제로 이동한 비용

* $h(n)$: 현재 Node n에서 목표 지점까지의 예상 비용 (Heuristics)

* $f(n)$: 시작 지점부터 목표 지점까지의 전체 예상 비용

* 장점:  빠른속도, 연산량, 메모리

* 단점:  비교적 덜 최적화된 경로

#### $3)$ 알고리즘 비교

|  | Dijkstra | A* |
|---|---|---|
| 현재까지의 비용 $g(n)$ | 사용 | 사용 |
| 목표까지의 예상 비용 $h(n)$ | 사용하지 않음 | 사용 |
| 목표 방향을 고려한 탐색 | X | O |
| 최단 경로 탐색 | O | △ |
| 일반적인 탐색 범위 | 넓음 | 상대적으로 좁음 |


각 알고리즘을 간단히 구현한 코드를 통해 비교를 해보도록 합시다.

In [ ]:
import heapq
import time


# 0: 빈 공간, 1: 벽/장애물
GRID_MAP = [
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 1, 1, 1, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    [0, 0, 1, 1, 1, 1, 0, 1, 0, 0],
    [0, 0, 1, 0, 0, 1, 0, 1, 0, 0],
    [0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, 1, 1, 0, 1, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
]  

# 시작 노드 Index
START = (0, 0)

# 목표 노드 Index
GOAL = (5, 4)


# 최종 결과 프린트
def draw_path(map_data, start, goal, path):
    for r in range(len(map_data)):
        row_str = ""
        for c in range(len(map_data[0])):
            if (r, c) == start: row_str += " S "        # 시작 노드
            elif (r, c) == goal: row_str += " G "       # 목표 노드
            elif (r, c) in path: row_str += " * "       # 경로
            elif map_data[r][c] == 1: row_str += " # "  # 벽
            else: row_str += " . "                      # 빈 공간
        print(row_str)


# 소요 시간 계산
def measure_time(algorithm, map_data, start, goal):
    start_time = time.perf_counter()
    _, _ = algorithm(map_data, start, goal)
    end_time = time.perf_counter()
    elapsed_time = end_time - start_time

    return elapsed_time

##### Dijkstra:

In [ ]:
# Dijkstra: f(n) = g(n)


def dijkstra(map_data, start, goal):
    neighbors = [(0, 1), (0, -1), (1, 0), (-1, 0)]
    g_score = {start: 0}
    came_from = {}      # 확정된 과거 노드 (최종 경로 역추적 용도)
    closed_set = set()  # Closed Set
    oheap = []          # Open Set (Heap 구조로 관리)

    # 시작 노드 추가
    heapq.heappush(oheap, (g_score[start], start))

    while oheap:
        # 현재까지 이동 비용이 가장 작은 노드 선택
        current = heapq.heappop(oheap)[1]

        # 목표에 도달한 경우 경로 역추적
        if current == goal:
            data = []

            while current in came_from:
                data.append(current)
                current = came_from[current]

            data.append(start)
            return data[::-1], len(closed_set)

        # 탐색 완료(예정)인 현재 노드를 Closed Set에 추가
        closed_set.add(current)

        # 현재 노드의 주변 노드 탐색
        for i, j in neighbors:
            neighbor = current[0] + i, current[1] + j
            g_score_tmp = g_score[current] + 1

            # 맵 범위를 벗어나거나 벽인 경우 스킵
            if 0 <= neighbor[0] < len(map_data):
                if 0 <= neighbor[1] < len(map_data[0]):
                    if map_data[neighbor[0]][neighbor[1]] == 1:
                        continue
                else: continue
            else: continue

            # 이미 탐색한 노드인 경우 스킵
            if neighbor in closed_set:
                continue

            # 현재 노드의 주변 노드를 Heap에 추가
            if g_score_tmp < g_score.get(neighbor, float("inf")):
                came_from[neighbor] = current
                g_score[neighbor] = g_score_tmp
                heapq.heappush(oheap, (g_score[neighbor], neighbor))

    return False, len(closed_set)


path_D, explored_node_count_D = dijkstra(GRID_MAP, START, GOAL)

##### A*:

In [ ]:
# A*: f(n) = g(n) + h(n)


def heuristic(a, b):
    # Manhattan 거리
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def astar(map_data, start, goal):
    neighbors = [(0, 1), (0, -1), (1, 0), (-1, 0)]
    g_score = {start: 0}
    f_score = {start: heuristic(start, goal)}
    came_from = {}      # 확정된 과거 노드 (최종 경로 역추적 용도)
    closed_set = set()  # Closed Set
    oheap = []          # Open Set (Heap 구조로 관리)

    # 시작 노드 추가
    heapq.heappush(oheap, (f_score[start], start))
    
    while oheap:
        # 현재까지 f(n) 비용이 가장 작은 노드 선택
        current = heapq.heappop(oheap)[1]

        # 목표에 도달한 경우 경로 역추적
        if current == goal:
            data = []

            while current in came_from:
                data.append(current)
                current = came_from[current]

            data.append(start)
            return data[::-1], len(closed_set)
        
        # 탐색 완료(예정)인 현재 노드를 Closed Set에 추가
        closed_set.add(current)

        # 현재 노드의 주변 노드 탐색
        for i, j in neighbors:
            neighbor = current[0] + i, current[1] + j
            g_score_tmp = g_score[current] + 1
            
            # 맵 범위를 벗어나거나 벽인 경우 스킵
            if 0 <= neighbor[0] < len(map_data):
                if 0 <= neighbor[1] < len(map_data[0]):
                    if map_data[neighbor[0]][neighbor[1]] == 1:
                        continue
                else: continue
            else: continue

            # 이미 탐색한 노드인 경우 스킵
            if neighbor in closed_set:
                continue

            # 현재 노드의 주변 노드를 Heap에 추가Behavior Tree
            if g_score_tmp < g_score.get(neighbor, float("inf")):
                came_from[neighbor] = current
                g_score[neighbor] = g_score_tmp
                f_score[neighbor] = g_score_tmp + heuristic(neighbor, goal)
                heapq.heappush(oheap, (f_score[neighbor], neighbor))
                
    return False, len(closed_set)


path_A, explored_node_count_A = astar(GRID_MAP, START, GOAL)

##### 결과 비교:

In [ ]:
draw_path(GRID_MAP, START, GOAL, path_D)

In [ ]:
draw_path(GRID_MAP, START, GOAL, path_A)

In [ ]:
print("===== Dijkstra =====")
print(f"실행 시간: {measure_time(dijkstra, GRID_MAP, START, GOAL):.8f}초")
print(f"탐색한 노드 수: {explored_node_count_D}개")

print()

print("======== A* ========")
print(f"실행 시간: {measure_time(astar, GRID_MAP, START, GOAL):.8f}초")
print(f"탐색한 노드 수: {explored_node_count_A}개")

---

## 4. Action

AMR이 주변 환경을 인식하고, 자신의 위치를 추정하고, 지도 작성 및 이동 경로까지 계획했다면, 마지막으로 실제 로봇의 움직임으로 실행해야 합니다. 계획된 경로에 따라 이동 명령을 생성하고 상황 변화에 맞춰 로봇 동작을 제어하는 과정을 **행동(Action)** 단계라고 합니다.

### **A. Local Planner/Controller**

이전 단계인 계획(Planning) 단계에서 Global Planner가 생성한 Global Path를 따라 로봇이 이동한다고 생각해 봅시다.

로봇은 Global Path를 따라 이동하면서 LiDAR 등의 센서를 이용해 주변 환경을 계속 확인합니다. 이때 Global Path는 이미 작성된 지도를 기반으로 생성되기 때문에, 아직 탐색하지 못한 지역의 장애물이나 움직이는 물체에는 대처하지 못합니다.

Global Path가 장애물을 고려하지 못하는 상황이라면, 로봇은 그대로 Global Path를 따라 이동해서는 안 됩니다. 주변의 실시간 센서 정보를 이용하여 장애물을 피할 수 있는 Local Path를 생성하고, 현재 상황에 맞는 이동 방향을 결정해야 합니다.

전체적인 흐름은 다음과 같습니다.

<div align="center">
Global Path<br>
↓<br>
주변 환경 관측<br>
↓<br>
현재 로봇이 따라갈 Local Path 생성<br>
↓<br>
로봇 바퀴에 속도 명령 전달<br>
↓<br>
로봇 제어
</div>

즉, **Global Planner**가 전체적인 이동 방향을 제시한다면, **Local Planner**는 현재 주변 상황을 고려하여 실제로 로봇이 이동할 경로를 결정합니다.

최종적으로 계산된 경로는 로봇의 **선속도/각속도** 혹은 **좌/우 바퀴 속도** 등의 이동 명령으로 변환되고, 이 명령이 로봇의 구동부에 전달되어 실제 움직임으로 이어집니다.

### **B. Path 기반 로봇 제어**

Path Planning에서 생성되는 경로는 일반적으로 연속적인 **Continuous** 곡선이 아니라 **Discrete** 형태인 Waypoint List로 표현됩니다.

예를 들어 다음과 같은 경로가 생성될 수 있습니다.

```python
path = [
    (1.0, 1.0),
    (1.5, 1.0),
    (2.0, 1.0),
    (2.5, 1.5),
    (3.0, 2.0)
]
```

각각의 $(x, y)$ 좌표가 로봇이 지나가야 할 Waypoint입니다. 즉, Planning 단계에서 만들어지는 Path는

<div align="center">
Path = [Waypoint 1, Waypoint 2, Waypoint 3, ...]
</div>

와 같은 Discrete한 데이터입니다.

하지만 실제 로봇은 이러한 점들을 하나씩 정확하게 방문하는 방식으로 움직이지 않습니다. 로봇의 움직임은 시간에 따라 위치와 방향이 계속해서 변하는 Continuous한 움직임입니다.<br>
따라서 Action 단계에서는 이러한 **Discrete Path**를 실제 로봇의 **Continuous한 움직임**으로 변환하는 과정이 필요합니다.

#### **Look-ahead**

Discrete한 Waypoint를 그대로 하나씩 따라가도록 로봇을 제어하면 여러 가지 문제가 발생할 수 있습니다.

예를 들어 로봇이 현재 Waypoint에 도착할 때마다 다음 Waypoint를 목표로 설정한다면, 로봇은 각각의 점에 도달하기 위해 반복적으로 방향을 변경할 수 있습니다.<br>
특히 로봇의 속도가 빠르거나 Waypoint 간격이 짧은 경우에는 로봇의 움직임이 부드럽지 않고 좌우로 크게 흔들리는 현상이 발생할 수 있습니다.

이를 해결하기 위해 로봇이 현재 위치에서 조금 앞쪽에 있는 Waypoint를 바라보도록 하는 방법을 사용할 수 있습니다.<br>
이를 **Look-ahead**라고 합니다.

##### **Look-ahead** 제어 과정

1. 로봇의 위치 파악
   * 현재 로봇 좌표 기준 가장 가까운 waypoint 계산
   * Euclidean Distance 사용

2. Look-ahead Point 설정
   * 거리 기반 목표점 설정
   * 직선 거리가 아닌 경로 거리

3. Look-ahead Point 좌표를 로봇 Frame으로 변환
   * 로봇 좌표를 원점으로 계산
   * 로봇 heading이 0° (+x축 방향)이 되도록 회전
   * 계산된 x값은 로봇과의 앞뒤 거리
   * 계산된 y값은 로봇과의 좌우 거리

4. 곡률 계산
   * 곡률: 경로가 휘어진 정도 ($m^{-1}$)
   * $\kappa > 0$: 좌측, $\kappa < 0$: 우측, $\kappa = 0$: 직선

5. 바퀴 속도 계산 및 전달
   * 선속도, 각속도 ($v, \omega$) 계산
   * 좌측, 우측 바퀴 속도 ($v_r, v_l$) 계산
   * 선속도($v$)는 로봇 default/max 속도로 설정 가능 (상수 취급)

##### 곡률 및 바퀴 속도 공식:

$$
\kappa = \frac{2y_{LA}}{x_{LA}^2 + y_{LA}^2}
\hspace{0.1cm},\hspace{0.6cm}
\omega = v\kappa
$$

<br>

$$
v = \frac{v_r + v_l}{2}
\hspace{0.1cm},\hspace{0.6cm}
\omega = \frac{v_r - v_l}{L}
$$

<br>

$$
v_r = v + \frac{\omega L}{2}
\hspace{0.1cm},\hspace{0.6cm}
v_l = v - \frac{\omega L}{2}
$$

##### Nearest Waypoint 계산 (경로 위 로봇의 위치 파악):

In [ ]:
from scipy.spatial import cKDTree


# Path의 Waypoints
path = [
    (1.0, 1.0),
    (1.5, 1.0),
    (2.0, 1.0),
    (2.5, 1.5),
    (3.0, 2.0),
    (3.5, 2.5)
]

# 현재 로봇 위치
robot_pos = (2.3, 1.4)

# Waypoint를 NumPy 배열로 변환
path_points = np.array(path)

# KD-Tree 생성
tree = cKDTree(path_points)

# 로봇 위치에서 가장 가까운 Waypoint 검색
distance, index = tree.query(robot_pos)
nearest_point = path[index]


print(f"로봇 위치: {robot_pos}")
print(f"가장 가까운 Waypoint: {nearest_point}")
print(f"Waypoint Index: {index}")
print(f"거리: {distance:.3f}")


In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(path_points[:, 0], path_points[:, 1], color="blue", s=60, label="Waypoints")
plt.scatter(robot_pos[0], robot_pos[1], color="red", s=150, marker="o", label="Robot")
plt.scatter(nearest_point[0], nearest_point[1], color="green", s=150, marker="*", label="Nearest Waypoint")
plt.plot(path_points[:, 0], path_points[:, 1], "b--", label="Path")
plt.plot([robot_pos[0], nearest_point[0]], [robot_pos[1], nearest_point[1]], "g-", linewidth=2, label=f"Distance = {distance:.3f}")
for i, (x, y) in enumerate(path) : plt.text(x + 0.05, y + 0.05, f"{i}", fontsize=10)

plt.title("Nearest Waypoint Search using cKDTree")
plt.xlabel("X"), plt.ylabel("Y"), plt.axis("equal"), plt.grid(True), plt.legend();

### **C. 상황 변화에 따른 행동 결정 (Decision-Making)**

실제 Search and Rescue 환경에서는 로봇이 단순히 Path만 따라가서는 충분하지 않습니다. 로봇은 이동하면서 계속해서 주변 상황을 확인하고, 상황에 따라 어떤 행동을 수행할지 결정해야 합니다.

예를 들어, 경로를 따라 주행하던 중 움직이는 장애물이 발견되면 장애물 회피 행동을 우선적으로 수행해야하며, 모든 목표 대상을 발견하여 귀환해야 하는 상황에는 귀환을 위한 새로운 경로를 재탐색하고 그에 맞는 바퀴 제어 명령을 생성해야 합니다.

또한, 위치 추정 오차가 크게 발생하여 현재 위치를 정확하게 파악할 수 없거나, 경로가 장애물로 막혀 더 이상 기존의 Global Path를 따라갈 수 없는 상황에서는 Recovery Action을 수행하여 문제를 해결하거나 새로운 주행을 시도해야 합니다.

이처럼 현재 상황을 판단하고 다음 행동을 결정하는 방법으로 대표적으로 **FSM(Finite State Machine)**과 **Behavior Tree**를 사용할 수 있습니다.

#### $1)$ FSM (Finite State Machine)

FSM은 로봇의 행동을 여러 개의 State로 나누고, 현재 상황에 따라 State를 전환하는 방식입니다.

예를 들어 Search and Rescue 로봇을 다음과 같이 구성할 수 있습니다.<br><br><br><br>

```text
                         목표 발견
         STATE="탐험"—————————————————STATE="접근"
              │          목표 구출          │
              │                           │
              │                           │
          전력 │ 충전                   충전 │ 모든 대상
          부족 │ 완료                   완료 │ 구출 완료
              │                           │
              │                           │
              │                           │
              └─────── STATE="귀환" ───────┘
```

<br><br><br>
각 State에서는 해당 상태에 필요한 행동을 수행하고, 특정 조건이 만족되면 다른 State로 전환(Transition)합니다.<br>
Transition의 조건으로는 거리, 시간, 센서 상태, 목표 탐지 여부 등 다양한 조건이 사용될 수 있습니다.

FSM은 구조가 단순하고 상태와 전환 조건을 명확하게 표현할 수 있다는 장점이 있습니다.<br>
하지만 로봇의 행동이 복잡해지고 상태와 전환 조건이 많아질수록 상태 간의 연결 관계가 복잡해지고, 전체 구조를 관리하고 수정하기 어려워지는 단점이 있습니다.

Finite State Machine, FSM 특징:

* 그래프 구조

* State(상태) & Transition(전이)

* 동작 프로세스가 단순하고 순차적인 경우

* 상태의 개수가 적고, 서로 복잡하게 얽힐 일이 없는 경우

* 상태와 조건/전이가 늘어날수록 복잡도가 기하급수적으로 증가


#### $2)$ Behavior Tree

Behavior Tree는 행동을 Tree 구조로 표현하여 로봇의 의사결정과 행동을 구성하는 방법입니다.<br>
FSM과 비교하면 Behavior Tree는 행동을 계층적인 구조로 표현하기 때문에 복잡한 로봇 행동을 여러 개의 작은 행동으로 나누어 구성하고 재사용하기에 적합합니다.

Behavior Tree 특징:

* 트리 구조

* Node로 구성

* 다양한 조건과 예외 처리가 복잡하게 얽힌 경우

* 조건에 따라 동작을 유연하게 선택하고 실행

* 노드의 추가·변경이 기존 로직에 미치는 영향이 적음

* 복잡한 행동과 예외 상황을 구조적으로 관리 가능

Behavior Tree의 노드는 크게 **실행 노드(Leaf Node)** 와 **제어 노드(Inner Node)** 로 나눌 수 있습니다.

실행 노드에는 특정 행동을 수행할 때 사용되는 **Task(Action)** 노드와 현재 상황이나 조건을 판단하는 **Condition** 노드가 있습니다.<br>
제어 노드는 자식 노드의 실행 흐름을 제어하며, 크게 **Selector**, **Sequence**, **Parallel**과 같은 **Composite Node**와 **Inverter**, **Loop**, **Repeater**와 같은 **Decorator Node**로 구성됩니다.

**실행 노드(Leaf Node)**:
* Task (Action)
* Condition

<br>

**제어 노드(Inner Node)**:
* Composite
  * Selector
  * Sequence
  * Parallel
* Decorator
  * Inverter
  * Loop
  * Repeater
  * Succeeder
  * Failer

##### **Selector**:
* 왼쪽 → 오른쪽 순서로 Child 실행

* [S]: 하나라도 성공한 경우

* [F]: 전부 실패한 경우

* [R]: 하나라도 Running인 경우

* 다음 Tick에서 Running Child부터 검사 (메모리 기반)


##### **Sequence:**

* 왼쪽 → 오른쪽 순서로 Child 실행

* [S]: 전부 성공한 경우

* [F]: 하나라도 실패한 경우

* [R]: 하나라도 Running인 경우

* 다음 Tick에서 Running Child부터 검사 (메모리 기반)


##### **Parallel:**

* 모든 Child 실행 (순서 의미 없음)

* [S]: 성공한 Child 수 >= Success Count

* [F]: 실패한 Child 수 >= Failure Count

* [R]: S/F 조건 충족하지 않은 경우

* 매 Tick마다 모든 Child 다시 실행/평가 (메모리 기반 X)


#### $3)$ Behavior Tree 코드 예시

##### **Behavior Tree의 구조 설계:**

```text
status.py
```

In [ ]:
from enum import Enum

class Status(Enum):
    SUCCESS = 1
    FAILURE = 2
    RUNNING = 3

```text
behavior_tree.py
```

In [ ]:
class BehaviorTree:
    def __init__(self, root):
        self.root = root

    def tick(self):
        return self.root.tick()

    def reset(self):
        self.root.reset()

```text
tree_node.py
```

In [ ]:
class Node:
    def __init__(self, name):
        self.name = name

    def tick(self):
        raise NotImplementedError

    def reset(self):
        pass

```text
composite.py
```

In [ ]:
class Composite(Node):
    def __init__(self, name):
        super().__init__(name)
        self.children = []

    def add_child(self, child):
        self.children.append(child)

    def reset(self):
        for child in self.children:
            child.reset()

```text
selector.py
```

In [ ]:
class Selector(Composite):
    def __init__(self, name):
        super().__init__(name)
        self.current_child = 0

    def tick(self):
        while self.current_child < len(self.children):
            status = self.children[self.current_child].tick()

            if status == Status.FAILURE:
                self.current_child += 1

            elif status == Status.RUNNING:
                return Status.RUNNING

            elif status == Status.SUCCESS:
                self.current_child = 0
                return Status.SUCCESS

        self.current_child = 0
        return Status.FAILURE

    def reset(self):
        super().reset()
        self.current_child = 0

```text
sequence.py
```

In [ ]:
class Sequence(Composite):
    def __init__(self, name):
        super().__init__(name)
        self.current_child = 0

    def tick(self):
        while self.current_child < len(self.children):
            status = self.children[self.current_child].tick()

            if status == Status.SUCCESS:
                self.current_child += 1

            elif status == Status.RUNNING:
                return Status.RUNNING

            elif status == Status.FAILURE:
                self.current_child = 0
                return Status.FAILURE

        self.current_child = 0
        return Status.SUCCESS

    def reset(self):
        super().reset()
        self.current_child = 0

```text
parallel.py
```

In [ ]:
class Parallel(Composite):
    def __init__(self, name, success_count=2):
        super().__init__(name)
        self.success_count = success_count

    def tick(self):
        statuses = []

        for child in self.children:
            statuses.append(child.tick())

        if Status.RUNNING in statuses:
            return Status.RUNNING

        success_num = statuses.count(Status.SUCCESS)

        if success_num >= self.success_count:
            return Status.SUCCESS

        return Status.FAILURE

```text
blackboard.py
```

In [ ]:
class Blackboard:
    def __init__(self):
        self.data = {}

    def set(self, key, value):
        self.data[key] = value

    def get(self, key, default=None):
        return self.data.get(key, default)

    def has(self, key):
        return key in self.data

    def remove(self, key):
        if key in self.data:
            del self.data[key]

    def clear(self):
        self.data.clear()

##### **테스트용 Leaf Nodes (실행 노드):**

In [ ]:
class AlwaysSuccess(Node):
    def __init__(self, name):
        super().__init__(name)

    def tick(self):
        print(f"{self.name}: SUCCESS")
        return Status.SUCCESS


class AlwaysFailure(Node):
    def __init__(self, name):
        super().__init__(name)

    def tick(self):
        print(f"{self.name}: FAILURE")
        return Status.FAILURE


class RunningThenSuccess(Node):
    def __init__(self, name, running_count=2):
        super().__init__(name)
        self.running_count = running_count
        self.count = 0

    def tick(self):
        if self.count < self.running_count:
            self.count += 1
            print(f"{self.name}: RUNNING ({self.count}/{self.running_count})")
            return Status.RUNNING

        print(f"{self.name}: SUCCESS")
        return Status.SUCCESS

    def reset(self):
        self.count = 0

##### **테스트:**

In [ ]:
print("\n****************** Sequence ******************")
print("\n========== Test 1: Sequence SUCCESS ==========")

seq = Sequence("Sequence1")
seq.add_child(AlwaysSuccess("A"))
seq.add_child(AlwaysSuccess("B"))
seq.add_child(AlwaysSuccess("C"))

tree = BehaviorTree(seq)

result = tree.tick()

print("Result:", result)
assert result == Status.SUCCESS


print("\n========== Test 2: Sequence FAILURE ==========")

seq = Sequence("Sequence2")
seq.add_child(AlwaysSuccess("A"))
seq.add_child(AlwaysFailure("B"))
seq.add_child(AlwaysSuccess("C"))

tree = BehaviorTree(seq)

result = tree.tick()

print("Result:", result)
assert result == Status.FAILURE


print("\n========== Test 3: Sequence RUNNING ==========")

seq = Sequence("Sequence3")
seq.add_child(AlwaysSuccess("A"))
seq.add_child(RunningThenSuccess("B", running_count=2))
seq.add_child(AlwaysSuccess("C"))

tree = BehaviorTree(seq)

for tick in range(3):
    print(f"\nTick {tick + 1}")

    result = tree.tick()

    print("Tree result:", result)

print("\n\n")
print("\n****************** Selector ******************")
print("\n========== Test 4: Selector SUCCESS ==========")

sel = Selector("Selector1")
sel.add_child(AlwaysFailure("A"))
sel.add_child(AlwaysFailure("B"))
sel.add_child(AlwaysSuccess("C"))

tree = BehaviorTree(sel)

result = tree.tick()

print("Result:", result)
assert result == Status.SUCCESS


print("\n========== Test 5: Selector FAILURE ==========")

sel = Selector("Selector2")
sel.add_child(AlwaysFailure("A"))
sel.add_child(AlwaysFailure("B"))
sel.add_child(AlwaysFailure("C"))

tree = BehaviorTree(sel)

result = tree.tick()

print("Result:", result)
assert result == Status.FAILURE


print("\n========== Test 6: Selector RUNNING ==========")

sel = Selector("Selector3")
sel.add_child(AlwaysFailure("A"))
sel.add_child(RunningThenSuccess("B", running_count=2))
sel.add_child(AlwaysFailure("C"))

tree = BehaviorTree(sel)

for tick in range(3):
    print(f"\nTick {tick + 1}")

    result = tree.tick()

    print("Tree result:", result)


print("\n\n")
print("\n****************** Parallel ******************")
print("\n========== Test 7: Parallel SUCCESS ==========")

parallel = Parallel("Parallel1")
parallel.add_child(AlwaysSuccess("A"))
parallel.add_child(AlwaysFailure("B"))
parallel.add_child(AlwaysSuccess("C"))

tree = BehaviorTree(parallel)

result = tree.tick()

print("Result:", result)
assert result == Status.SUCCESS


print("\n========== Test 8: Parallel FAILURE ==========")

parallel = Parallel("Parallel2")
parallel.add_child(AlwaysSuccess("A"))
parallel.add_child(AlwaysFailure("B"))
parallel.add_child(AlwaysFailure("C"))

tree = BehaviorTree(parallel)

result = tree.tick()

print("Result:", result)
assert result == Status.FAILURE


print("\n========== Test 9: Parallel RUNNING ==========")

parallel = Parallel("Parallel3")
parallel.add_child(AlwaysSuccess("A"))
parallel.add_child(RunningThenSuccess("B", running_count=2))
parallel.add_child(AlwaysSuccess("C"))

tree = BehaviorTree(parallel)

for tick in range(3):
    print(f"\nTick {tick + 1}")

    result = tree.tick()

    print("Tree result:", result)

---

## 5. Webots 시뮬레이션

### **설치**

```bash
cd ~/Downloads
wget https://github.com/cyberbotics/webots/releases/download/R2025a/webots_2025a_amd64.deb
sudo apt install ./webots_2025a_amd64.deb
```

### 핵심 파라미터

```python
WHEEL_RADIUS = 0.033
WHEEL_SEPARATION = 0.160
ROBOT_RADIUS = 0.105
```

### **시뮬레이션 시연**

Webots 시뮬레이션은 직접 시연을 통해 설명드리도록 하겠습니다.

---
---

<br><br><div style="text-align: right; color: gray; font-style: italic;">
© 2026 김규래 (Kyu Rae Kim). All rights reserved.&emsp;<br><br>
This material is provided solely for the intended instructional purpose.&emsp;<br>
Redistribution, reproduction, modification, adaptation, or reuse of this material in any form without prior written permission from the copyright holder is prohibited.&emsp;
</div>